# Training Model Kata WL-BISINDO untuk SAPA

Melatih model pengenalan isyarat kata-level (32 kelas) dari dataset publik
[WL-BISINDO](https://www.kaggle.com/datasets/glennleonali/wl-bisindo) (CC BY-NC 4.0),
lalu mengekspor ke format TensorFlow.js untuk inferensi di browser.

**Sebelum menjalankan:**
1. Attach dataset `glennleonali/wl-bisindo` (panel kanan, Add Input)
2. Opsional tapi sangat disarankan: attach juga dataset berisi `word_seqs.json`
   hasil ekstraksi lokal, supaya ekstraksi MediaPipe (~1 jam) dilewati
3. Nyalakan GPU: Settings > Accelerator > GPU T4 x2 (atau P100)

Arsitektur dan fitur identik dengan `scripts/train_words.mjs` dan
`src/lib/word-features.ts` di repo SAPA: 48 frame x 36 fitur per sekuens.

In [ ]:
# Cell 1: konfigurasi dan pencarian input
import glob, json, math, os, re

import numpy as np

WINDOW = 48
FEAT = 36
TARGET_FPS = 15
EPOCHS = 60

WORD_LABELS = {
    "0": "air", "1": "belajar", "2": "cari", "3": "hari", "4": "ingat",
    "5": "lagi", "6": "maaf", "7": "makan", "8": "motor", "9": "saya",
    "10": "terimakasih", "11": "tuli", "12": "apa", "13": "siapa",
    "14": "kapan", "15": "dimana", "16": "mengapa", "17": "bagaimana",
    "18": "merah", "19": "kuning", "20": "hijau", "21": "hitam",
    "22": "dengar", "23": "berangkat", "24": "datang", "25": "teman",
    "26": "keluarga", "27": "rumah", "28": "pagi", "29": "siang",
    "30": "sore", "31": "malam",
}

# cari word_seqs.json di input yang ter-attach (jalur cepat)
seq_candidates = glob.glob("/kaggle/input/**/word_seqs.json", recursive=True)
SEQ_PATH = seq_candidates[0] if seq_candidates else None

# cari folder video WL-BISINDO (jalur lambat, ekstraksi ulang)
video_files = glob.glob("/kaggle/input/**/signer*_label*_sample*.mp4", recursive=True)
print(f"word_seqs.json: {SEQ_PATH}")
print(f"video ditemukan: {len(video_files)}")
assert SEQ_PATH or video_files, "Attach dataset wl-bisindo atau word_seqs.json dulu"

In [ ]:
# Cell 2: muat fitur (jalur cepat) atau ekstrak ulang dengan MediaPipe (lambat)
CHAINS = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16], [17, 18, 19, 20]]

def _curls(pts):
    out = []
    for chain in CHAINS:
        joints = [pts[0]] + [pts[i] for i in chain]
        for j in range(1, 4):
            v1 = joints[j] - joints[j - 1]
            v2 = joints[j + 1] - joints[j]
            d1, d2 = np.linalg.norm(v1), np.linalg.norm(v2)
            if d1 < 1e-6 or d2 < 1e-6:
                out.append(0.0)
            else:
                c = float(np.dot(v1, v2) / (d1 * d2))
                out.append(math.acos(max(-1.0, min(1.0, c))))
    return out

def _frame_features(result):
    feat = np.zeros(36, dtype=np.float32)
    hands = {}
    for lms, handed in zip(result.hand_landmarks, result.handedness):
        label, score = handed[0].category_name, handed[0].score
        if label not in hands or score > hands[label][1]:
            pts = np.array([[p.x, p.y, p.z] for p in lms], dtype=np.float32)
            hands[label] = (pts, score)
    if "Left" in hands:
        pts = hands["Left"][0]
        feat[0:15] = _curls(pts); feat[30] = 1
        feat[32] = pts[0][0] - 0.5; feat[33] = pts[0][1] - 0.5
    if "Right" in hands:
        pts = hands["Right"][0]
        feat[15:30] = _curls(pts); feat[31] = 1
        feat[34] = pts[0][0] - 0.5; feat[35] = pts[0][1] - 0.5
    return feat

if SEQ_PATH:
    with open(SEQ_PATH) as f:
        seqs = json.load(f)
    print(f"Fitur dimuat dari {SEQ_PATH}: {len(seqs)} sekuens")
else:
    # ekstraksi ulang di Kaggle (~1 jam, CPU): perlu mediapipe
    %pip install -q mediapipe opencv-python
    import cv2
    import mediapipe as mp
    import urllib.request
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision

    task_path = "/kaggle/working/hand_landmarker.task"
    if not os.path.exists(task_path):
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
            task_path,
        )
    base = mp_python.BaseOptions(model_asset_path=task_path)
    opts = vision.HandLandmarkerOptions(
        base_options=base, num_hands=2,
        min_hand_detection_confidence=0.3,
        running_mode=vision.RunningMode.VIDEO,
    )
    pat = re.compile(r"signer(\d+)_label(\d+)_sample(\d+)\.mp4$")
    seqs = []
    for vi, path in enumerate(sorted(video_files)):
        m = pat.search(os.path.basename(path))
        if not m:
            continue
        detector = vision.HandLandmarker.create_from_options(opts)
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        step = max(1, round(fps / TARGET_FPS))
        seq, idx, ts = [], 0, 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if idx % step == 0:
                h, w = frame.shape[:2]
                if max(h, w) > 480:
                    s = 480 / max(h, w)
                    frame = cv2.resize(frame, (int(w * s), int(h * s)))
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                ts += int(1000 / TARGET_FPS)
                seq.append(_frame_features(detector.detect_for_video(mp_img, ts)).tolist())
            idx += 1
        cap.release(); detector.close()
        seqs.append({"signer": m.group(1), "label": m.group(2), "sample": m.group(3),
                     "fps": TARGET_FPS, "seq": seq, "file": os.path.basename(path)})
        if (vi + 1) % 100 == 0:
            print(f"{vi + 1}/{len(video_files)}", flush=True)
            with open("/kaggle/working/word_seqs.partial.json", "w") as f:
                json.dump(seqs, f)
    with open("/kaggle/working/word_seqs.json", "w") as f:
        json.dump(seqs, f)
    print(f"Ekstraksi selesai: {len(seqs)} sekuens")

In [ ]:
# Cell 3: siapkan data (resample 48 frame, split signer-independent, augmentasi)
labels = sorted({s["label"] for s in seqs}, key=int)
label_idx = {l: i for i, l in enumerate(labels)}
words = [WORD_LABELS.get(l, f"kata-{l}") for l in labels]
print(f"{len(seqs)} sekuens, {len(labels)} kelas: {words}")

def resample(seq, n=WINDOW):
    seq = np.asarray(seq, dtype=np.float32)
    idx = np.linspace(0, len(seq) - 1, n)
    lo = np.floor(idx).astype(int)
    hi = np.minimum(lo + 1, len(seq) - 1)
    a = (idx - lo)[:, None]
    return seq[lo] * (1 - a) + seq[hi] * a

def mirror(frames):
    g = frames.copy()
    g[:, 0:15], g[:, 15:30] = frames[:, 15:30], frames[:, 0:15]
    g[:, 30], g[:, 31] = frames[:, 31], frames[:, 30]
    g[:, 32] = -frames[:, 34]; g[:, 33] = frames[:, 35]
    g[:, 34] = -frames[:, 32]; g[:, 35] = frames[:, 33]
    return g

def jitter(seq):
    n = len(seq)
    cs = int(n * np.random.rand() * 0.12)
    ce = int(n * np.random.rand() * 0.12)
    sliced = seq[cs:n - ce]
    return resample(sliced) if len(sliced) >= 8 else resample(seq)

signers = sorted({s["signer"] for s in seqs})
val_signer = signers[-1]
print(f"Signer: {signers}, validasi: {val_signer} (signer-independent)")

x_train, y_train, x_val, y_val = [], [], [], []
skipped = 0
for s in seqs:
    arr = np.asarray(s["seq"], dtype=np.float32)
    if len(arr) < 8 or not ((arr[:, 30] > 0) | (arr[:, 31] > 0)).any():
        skipped += 1
        continue
    base = resample(arr)
    yi = label_idx[s["label"]]
    if s["signer"] == val_signer:
        x_val.append(base); y_val.append(yi)
    else:
        x_train += [base, mirror(base), jitter(arr)]
        y_train += [yi, yi, yi]

x_train = np.stack(x_train); y_train = np.array(y_train)
x_val = np.stack(x_val); y_val = np.array(y_val)
print(f"Train {x_train.shape}, Val {x_val.shape}, dilewati {skipped}")

In [ ]:
# Cell 4: bangun dan latih model (arsitektur sama dengan train_words.mjs)
import tensorflow as tf
from tensorflow.keras import layers

print("GPU:", tf.config.list_physical_devices("GPU"))

model = tf.keras.Sequential([
    layers.Input(shape=(WINDOW, FEAT)),
    layers.Conv1D(64, 5, strides=2, activation="relu"),
    layers.Dropout(0.3),
    layers.Conv1D(128, 3, strides=2, activation="relu"),
    layers.Dropout(0.3),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(len(labels), activation="softmax"),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

ckpt = tf.keras.callbacks.ModelCheckpoint(
    "/kaggle/working/best.keras", monitor="val_accuracy",
    save_best_only=True, verbose=0,
)
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS, batch_size=32, shuffle=True,
    callbacks=[ckpt], verbose=2,
)

model = tf.keras.models.load_model("/kaggle/working/best.keras")
val_loss, val_acc = model.evaluate(x_val, y_val, verbose=0)
print(f"\nAkurasi validasi signer-independent (checkpoint terbaik): {val_acc * 100:.2f}%")

preds = model.predict(x_val, verbose=0).argmax(-1)
print("\nAkurasi per kata:")
for i, w in enumerate(words):
    mask = y_val == i
    if mask.any():
        acc = (preds[mask] == i).mean() * 100
        print(f"  {w}: {acc:.0f}% ({(preds[mask] == i).sum()}/{mask.sum()})")

In [ ]:
# Cell 5: ekspor ke TensorFlow.js, siap dipakai langsung oleh app SAPA
%pip install -q tensorflowjs
import json as _json

out_dir = "/kaggle/working/bisindo-words"
!rm -rf {out_dir} && mkdir -p {out_dir}

# simpan format H5 lalu konversi (kompatibel tensorflowjs_converter)
model.save("/kaggle/working/best.h5")
!tensorflowjs_converter --input_format=keras /kaggle/working/best.h5 {out_dir}

with open(f"{out_dir}/metadata.json", "w") as f:
    _json.dump({
        "words": words,
        "window": WINDOW,
        "featureSize": FEAT,
        "valAccuracy": float(val_acc),
        "valSigner": val_signer,
        "dataset": "kaggle:glennleonali/wl-bisindo (CC BY-NC 4.0)",
        "trainedOn": "kaggle-gpu",
    }, f, indent=2)

!ls -la {out_dir}
!cd /kaggle/working && zip -r bisindo-words-tfjs.zip bisindo-words
print("\nSelesai. Download bisindo-words-tfjs.zip dari panel Output,")
print("ekstrak isinya ke public/models/bisindo-words/ di project SAPA.")